# Universal Uploader Server

### Run this notebook and keep it open. Send URLs from your local machine.

**How it works:**
1. Run all cells below (just once)
2. The server starts monitoring for new URLs
3. On your local machine, run: `python local_client.py "https://example.com/file.zip"`
4. This notebook automatically downloads and uploads to your Drive

---

In [ ]:
#@title 1. Setup & Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
%pip install -q yt-dlp requests tqdm

print("Setup complete!")

In [ ]:
#@title 2. Initialize Server Engine

import os
import json
import time
import re
import mimetypes
from datetime import datetime
from urllib.parse import urlparse, unquote

import requests
from tqdm.notebook import tqdm

# ==================== CONFIGURATION ====================
class Config:
    """Server configuration."""
    DRIVE_BASE = "/content/drive/MyDrive"
    QUEUE_FOLDER = "UploaderQueue"
    QUEUE_FILE = "queue.json"
    OUTPUT_FOLDER = "Downloads"
    CHUNK_SIZE = 10 * 1024 * 1024
    MAX_RETRIES = 5
    POLL_INTERVAL = 5
    USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"


# Create queue folder
QUEUE_PATH = os.path.join(Config.DRIVE_BASE, Config.QUEUE_FOLDER)
QUEUE_FILE_PATH = os.path.join(QUEUE_PATH, Config.QUEUE_FILE)
os.makedirs(QUEUE_PATH, exist_ok=True)


# ==================== UTILITY FUNCTIONS ====================
def format_size(size_bytes):
    """Convert bytes to human-readable format."""
    if size_bytes is None or size_bytes == 0:
        return "0 B"
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} PB"


def format_speed(bps):
    """Convert bytes/sec to human-readable format."""
    return format_size(bps) + "/s" if bps else "0 B/s"


def sanitize_filename(filename):
    """Remove invalid characters from filename."""
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    return filename.strip(' .')[:200] or "downloaded_file"


def get_filename_from_url(url, response=None):
    """Extract filename from URL or response headers."""
    if response and 'Content-Disposition' in response.headers:
        cd = response.headers['Content-Disposition']
        matches = re.findall(
            r'filename[*]?=["\']?(?:UTF-8\'\')?(.[^"\';<>]+)',
            cd, re.IGNORECASE
        )
        if matches:
            return sanitize_filename(unquote(matches[0]))
    parsed = urlparse(url)
    filename = os.path.basename(unquote(parsed.path))
    if filename and '.' in filename:
        return sanitize_filename(filename)
    return None


def detect_url_type(url):
    """Detect the type of URL for appropriate handling."""
    url_lower = url.lower()
    video_domains = [
        'youtube.com', 'youtu.be', 'twitter.com', 'x.com',
        'instagram.com', 'tiktok.com', 'reddit.com', 'twitch.tv',
        'vimeo.com', 'facebook.com', 'fb.watch'
    ]
    for domain in video_domains:
        if domain in url_lower:
            return 'video'
    if 'mega.nz' in url_lower:
        return 'mega'
    return 'direct'


# ==================== DOWNLOAD FUNCTIONS ====================
def download_direct(url, save_path, filename=None):
    """Download file directly with retry logic."""
    headers = {'User-Agent': Config.USER_AGENT}

    for attempt in range(Config.MAX_RETRIES):
        try:
            response = requests.get(
                url, headers=headers, stream=True,
                allow_redirects=True, timeout=60
            )
            response.raise_for_status()

            if not filename:
                filename = get_filename_from_url(url, response)
            if not filename:
                ext = mimetypes.guess_extension(
                    response.headers.get('Content-Type', '').split(';')[0]
                ) or '.bin'
                filename = f"download_{int(time.time())}{ext}"

            filename = sanitize_filename(filename)
            filepath = os.path.join(save_path, filename)

            total_size = int(response.headers.get('content-length', 0))
            downloaded = 0
            start_time = time.time()

            with open(filepath, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True,
                          desc=filename[:30]) as pbar:
                    for chunk in response.iter_content(chunk_size=Config.CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
                            pbar.update(len(chunk))

            duration = time.time() - start_time
            size = os.path.getsize(filepath)

            return {
                'success': True,
                'filepath': filepath,
                'filename': filename,
                'size': size,
                'duration': duration,
                'speed': size / duration if duration > 0 else 0
            }

        except requests.RequestException as req_err:
            if attempt < Config.MAX_RETRIES - 1:
                print(f"   Attempt {attempt+1} failed: {req_err}. Retrying...")
                time.sleep(3 * (attempt + 1))
            else:
                return {'success': False, 'error': str(req_err)}
    return {'success': False, 'error': 'Max retries exceeded'}


def download_ytdlp(url, save_path, format_spec='best'):
    """Download using yt-dlp for video sites."""
    try:
        import yt_dlp

        outtmpl = os.path.join(save_path, '%(title)s.%(ext)s')

        ydl_opts = {
            'format': format_spec,
            'outtmpl': outtmpl,
            'quiet': True,
            'no_warnings': True,
            'retries': Config.MAX_RETRIES,
        }

        start_time = time.time()

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            if info:
                if 'requested_downloads' in info:
                    filepath = info['requested_downloads'][0]['filepath']
                else:
                    filepath = ydl.prepare_filename(info)

                duration = time.time() - start_time
                size = os.path.getsize(filepath) if os.path.exists(filepath) else 0

                return {
                    'success': True,
                    'filepath': filepath,
                    'filename': os.path.basename(filepath),
                    'size': size,
                    'duration': duration,
                    'speed': size / duration if duration > 0 else 0
                }

        return {'success': False, 'error': 'yt-dlp extraction failed'}

    except ImportError:
        return {'success': False, 'error': 'yt-dlp not installed'}
    except Exception as err:
        return {'success': False, 'error': str(err)}


# ==================== QUEUE MANAGEMENT ====================
def load_queue():
    """Load queue from file."""
    if os.path.exists(QUEUE_FILE_PATH):
        try:
            with open(QUEUE_FILE_PATH, 'r', encoding='utf-8') as f:
                return json.load(f)
        except (json.JSONDecodeError, IOError):
            return {'items': [], 'processed': []}
    return {'items': [], 'processed': []}


def save_queue(queue_data):
    """Save queue to file."""
    with open(QUEUE_FILE_PATH, 'w', encoding='utf-8') as f:
        json.dump(queue_data, f, indent=2)


def process_item(item):
    """Process a single queue item."""
    url = item.get('url')
    filename = item.get('filename')
    folder = item.get('folder', Config.OUTPUT_FOLDER)
    method = item.get('method', 'auto')
    video_format = item.get('format', 'best')

    # Create output folder
    save_path = os.path.join(Config.DRIVE_BASE, folder)
    os.makedirs(save_path, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"PROCESSING: {url[:50]}...")
    print(f"{'='*60}")

    # Auto-detect method
    if method == 'auto':
        method = detect_url_type(url)

    # Download
    if method == 'video':
        result = download_ytdlp(url, save_path, video_format)
    else:
        result = download_direct(url, save_path, filename)

    # Print result
    if result.get('success'):
        print("\nSUCCESS!")
        print(f"   File: {result.get('filename')}")
        print(f"   Size: {format_size(result.get('size', 0))}")
        print(f"   Speed: {format_speed(result.get('speed', 0))}")
        print(f"   Path: {result.get('filepath')}")
    else:
        print(f"\nFAILED: {result.get('error')}")

    return result


print("Server engine initialized!")
print(f"\nQueue folder: {QUEUE_PATH}")
print(f"Queue file: {QUEUE_FILE_PATH}")

In [ ]:
#@title 3. Start Server (Run this and keep it running)

from IPython.display import clear_output, display
import ipywidgets as widgets

# Status display
status_output = widgets.Output()


def update_status(message, is_processing=False):
    """Update status display."""
    with status_output:
        clear_output(wait=True)
        prefix = "Processing" if is_processing else "Waiting"
        print(f"[{prefix}] {message}")


print("=" * 60)
print("       UNIVERSAL UPLOADER SERVER STARTED")
print("=" * 60)
print("  The server is now monitoring for new URLs.")
print("")
print("  On your local machine, run:")
print('  python src/local_client.py "https://example.com/file"')
print("")
print("  Press the STOP button to stop the server.")
print("=" * 60)
print()

display(status_output)

processed_ids = set()
total_processed = 0

try:
    while True:
        # Load queue
        queue = load_queue()
        items = queue.get('items', [])

        # Find pending items
        pending = [
            item for item in items
            if item.get('id') not in processed_ids
            and item.get('status') == 'pending'
        ]

        if pending:
            for item in pending:
                item_id = item.get('id')

                # Mark as processing
                item['status'] = 'processing'
                item['started_at'] = datetime.now().isoformat()
                save_queue(queue)

                url_preview = item.get('url', '')[:40]
                update_status(f"{url_preview}...", True)

                # Process
                result = process_item(item)

                # Update status
                item['status'] = 'completed' if result.get('success') else 'failed'
                item['completed_at'] = datetime.now().isoformat()
                item['result'] = {
                    'success': result.get('success'),
                    'filename': result.get('filename'),
                    'size': result.get('size'),
                    'filepath': result.get('filepath'),
                    'error': result.get('error')
                }
                save_queue(queue)

                processed_ids.add(item_id)
                total_processed += 1

                print(f"\nTotal processed: {total_processed}")

        update_status(f"Waiting for URLs... (Processed: {total_processed})")
        time.sleep(Config.POLL_INTERVAL)

except KeyboardInterrupt:
    print("\n\nServer stopped by user.")
    print(f"Total files processed: {total_processed}")

---

## Manual Operations (Optional)

Use these cells for manual control if needed.

In [ ]:
#@title View Queue Status

queue = load_queue()
items = queue.get('items', [])

print("QUEUE STATUS")
print("=" * 60)

if not items:
    print("   Queue is empty.")
else:
    for item in items[-10:]:
        status_icons = {
            'pending': '[PENDING]',
            'processing': '[PROCESSING]',
            'completed': '[DONE]',
            'failed': '[FAILED]'
        }
        status_icon = status_icons.get(item.get('status'), '[?]')

        url = item.get('url', '')[:40]
        print(f"   {status_icon} {url}...")
        if item.get('result', {}).get('filename'):
            print(f"      -> {item['result']['filename']}")

print("=" * 60)
pending = len([i for i in items if i.get('status') == 'pending'])
completed = len([i for i in items if i.get('status') == 'completed'])
failed = len([i for i in items if i.get('status') == 'failed'])
print(f"   Pending: {pending} | Completed: {completed} | Failed: {failed}")

In [ ]:
#@title Clear Queue

CONFIRM_CLEAR = False  #@param {type:"boolean"}

if CONFIRM_CLEAR:
    save_queue({'items': [], 'processed': []})
    print("Queue cleared!")
else:
    print("Set CONFIRM_CLEAR to True to clear the queue.")

In [ ]:
#@title Manual Single Download (without queue)

URL = "https://example.com/file.zip"  #@param {type:"string"}
FOLDER = "Downloads"  #@param {type:"string"}
USE_YTDLP = False  #@param {type:"boolean"}

item = {
    'url': URL,
    'folder': FOLDER,
    'method': 'video' if USE_YTDLP else 'auto'
}

result = process_item(item)